# TPC-DS Dataset Exploration

This notebook demonstrates how to work with the TPC-DS e-commerce dataset generated in DuckDB.

**Dataset**: TPC-DS Scale Factor 1 (~1GB)
**Tables**: 24 tables modeling an e-commerce business
**Location**: `../data/ecommerce_dw.duckdb`

## 1. Connect to Database

In [1]:
import duckdb
import pandas as pd
import os

# Dynamic path resolution
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
DB_PATH = os.path.join(BASE_DIR, 'data', 'ecommerce_dw.duckdb')

print(f"Connecting to: {DB_PATH}")
con = duckdb.connect(database=DB_PATH, read_only=True)
print("✓ Connected successfully!")

Connecting to: /media/zeronista/New Volume/FPT/S9/ISP490/Capstone-NLUS-VDD/data/ecommerce_dw.duckdb
✓ Connected successfully!


## 2. Explore Schema

Let's see what tables are available.

In [2]:
# List all tables
tables = con.sql("SHOW TABLES").df()
print(f"Found {len(tables)} tables:\n")
tables

Found 24 tables:



,name
0,call_center
1,catalog_page
2,catalog_returns
3,catalog_sales
4,customer
5,customer_address
6,customer_demographics
7,date_dim
8,household_demographics
9,income_band


## 3. Get Schema for Specific Tables

Let's examine the structure of key tables.

In [3]:
def show_table_info(table_name):
    """Display table schema and row count."""
    print(f"\n{'='*60}")
    print(f"TABLE: {table_name}")
    print(f"{'='*60}")
    
    # Get schema
    schema = con.sql(f"DESCRIBE {table_name}").df()
    print(f"\nColumns ({len(schema)}):")
    print(schema[['column_name', 'column_type']].to_string(index=False))
    
    # Get row count
    count = con.sql(f"SELECT COUNT(*) as count FROM {table_name}").fetchone()[0]
    print(f"\nRow Count: {count:,}")
    
# Show info for key tables
key_tables = ['customer', 'item', 'store_sales', 'date_dim']
for table in key_tables:
    show_table_info(table)


TABLE: customer

Columns (18):
           column_name column_type
         c_customer_sk      BIGINT
         c_customer_id     VARCHAR
    c_current_cdemo_sk      BIGINT
    c_current_hdemo_sk      BIGINT
     c_current_addr_sk      BIGINT
c_first_shipto_date_sk      BIGINT
 c_first_sales_date_sk      BIGINT
          c_salutation     VARCHAR
          c_first_name     VARCHAR
           c_last_name     VARCHAR
 c_preferred_cust_flag     VARCHAR
           c_birth_day      BIGINT
         c_birth_month      BIGINT
          c_birth_year      BIGINT
       c_birth_country     VARCHAR
               c_login     VARCHAR
       c_email_address     VARCHAR
 c_last_review_date_sk     INTEGER

Row Count: 100,000

TABLE: item

Columns (22):
     column_name  column_type
       i_item_sk       BIGINT
       i_item_id      VARCHAR
i_rec_start_date         DATE
  i_rec_end_date         DATE
     i_item_desc      VARCHAR
 i_current_price DECIMAL(7,2)
i_wholesale_cost DECIMAL(7,2)
      i_brand_i

## 4. Giới hạn và Phạm vi Dữ liệu (Data Limits & Ranges)

Phân tích các giới hạn và phạm vi dữ liệu trong dataset để hiểu rõ quy mô và đặc điểm của dữ liệu.


In [4]:
# Chức năng: Hiển thị các giới hạn và phạm vi dữ liệu quan trọng của dataset
# Cách hoạt động: Thực hiện các truy vấn để lấy min/max/count của các trường quan trọng

print("="*80)
print("TỔNG QUAN VỀ GIỚI HẠN DỮ LIỆU TPC-DS DATASET")
print("="*80)

# 1. Phạm vi thời gian của dữ liệu
print("\n1. PHẠM VI THỜI GIAN (Date Ranges)")
print("-" * 60)
date_range = con.sql("""
    SELECT 
        MIN(d_date) as earliest_date,
        MAX(d_date) as latest_date,
        COUNT(DISTINCT d_date) as total_days,
        MIN(d_year) as start_year,
        MAX(d_year) as end_year,
        MAX(d_year) - MIN(d_year) + 1 as year_span
    FROM date_dim
""").df()
print(date_range.to_string(index=False))

# 2. Phạm vi giá sản phẩm
print("\n\n2. PHẠM VI GIÁ SẢN PHẨM (Price Ranges)")
print("-" * 60)
price_range = con.sql("""
    SELECT 
        MIN(i_current_price) as min_price,
        MAX(i_current_price) as max_price,
        AVG(i_current_price) as avg_price,
        MEDIAN(i_current_price) as median_price,
        MIN(i_wholesale_cost) as min_wholesale_cost,
        MAX(i_wholesale_cost) as max_wholesale_cost,
        AVG(i_wholesale_cost) as avg_wholesale_cost
    FROM item
""").df()
print(price_range.to_string(index=False))

# 3. Phạm vi giao dịch bán hàng
print("\n\n3. PHẠM VI GIAO DỊCH BÁN HÀNG (Sales Transaction Ranges)")
print("-" * 60)
sales_range = con.sql("""
    SELECT 
        MIN(ss_quantity) as min_quantity,
        MAX(ss_quantity) as max_quantity,
        AVG(ss_quantity) as avg_quantity,
        MIN(ss_sales_price) as min_sales_price,
        MAX(ss_sales_price) as max_sales_price,
        AVG(ss_sales_price) as avg_sales_price,
        MIN(ss_net_paid) as min_transaction_value,
        MAX(ss_net_paid) as max_transaction_value,
        AVG(ss_net_paid) as avg_transaction_value
    FROM store_sales
""").df()
print(sales_range.to_string(index=False))

# 4. Số lượng bản ghi trong các bảng chính
print("\n\n4. SỐ LƯỢNG BẢN GHI TRONG CÁC BẢNG CHÍNH (Record Counts)")
print("-" * 60)
table_counts = con.sql("""
    SELECT 'customer' as table_name, COUNT(*) as row_count FROM customer
    UNION ALL
    SELECT 'item' as table_name, COUNT(*) as row_count FROM item
    UNION ALL
    SELECT 'store' as table_name, COUNT(*) as row_count FROM store
    UNION ALL
    SELECT 'store_sales' as table_name, COUNT(*) as row_count FROM store_sales
    UNION ALL
    SELECT 'web_sales' as table_name, COUNT(*) as row_count FROM web_sales
    UNION ALL
    SELECT 'catalog_sales' as table_name, COUNT(*) as row_count FROM catalog_sales
    UNION ALL
    SELECT 'store_returns' as table_name, COUNT(*) as row_count FROM store_returns
    UNION ALL
    SELECT 'web_returns' as table_name, COUNT(*) as row_count FROM web_returns
    UNION ALL
    SELECT 'catalog_returns' as table_name, COUNT(*) as row_count FROM catalog_returns
    ORDER BY row_count DESC
""").df()
print(table_counts.to_string(index=False))

# 5. Phạm vi dữ liệu khách hàng
print("\n\n5. PHẠM VI DỮ LIỆU KHÁCH HÀNG (Customer Data Ranges)")
print("-" * 60)
customer_range = con.sql("""
    SELECT 
        COUNT(DISTINCT c_customer_sk) as total_customers,
        MIN(c_birth_year) as min_birth_year,
        MAX(c_birth_year) as max_birth_year,
        COUNT(DISTINCT c_birth_country) as countries_count,
        COUNT(CASE WHEN c_email_address IS NOT NULL THEN 1 END) as customers_with_email,
        COUNT(DISTINCT c_current_addr_sk) as unique_addresses
    FROM customer
""").df()
print(customer_range.to_string(index=False))

# 6. Phạm vi danh mục sản phẩm
print("\n\n6. PHÂN LOẠI SẢN PHẨM (Product Categories)")
print("-" * 60)
product_categories = con.sql("""
    SELECT 
        COUNT(DISTINCT i_category) as total_categories,
        COUNT(DISTINCT i_class) as total_classes,
        COUNT(DISTINCT i_brand) as total_brands,
        COUNT(DISTINCT i_manufact) as total_manufacturers,
        COUNT(DISTINCT i_color) as total_colors,
        COUNT(DISTINCT i_size) as total_sizes
    FROM item
""").df()
print(product_categories.to_string(index=False))

# 7. Phạm vi cửa hàng
print("\n\n7. PHẠM VI CỬA HÀNG (Store Information)")
print("-" * 60)
store_info = con.sql("""
    SELECT 
        COUNT(DISTINCT s_store_sk) as total_stores,
        COUNT(DISTINCT s_state) as states_count,
        COUNT(DISTINCT s_city) as cities_count,
        MIN(s_number_employees) as min_employees,
        MAX(s_number_employees) as max_employees,
        AVG(s_number_employees) as avg_employees
    FROM store
""").df()
print(store_info.to_string(index=False))

# 8. Tổng dung lượng dữ liệu
print("\n\n8. TỔNG DUNG LƯỢNG DATASET (Dataset Size)")
print("-" * 60)
total_sales = con.sql("""
    SELECT 
        (SELECT COUNT(*) FROM store_sales) + 
        (SELECT COUNT(*) FROM web_sales) + 
        (SELECT COUNT(*) FROM catalog_sales) as total_sale_transactions,
        (SELECT COUNT(*) FROM store_returns) + 
        (SELECT COUNT(*) FROM web_returns) + 
        (SELECT COUNT(*) FROM catalog_returns) as total_return_transactions
""").df()
print(total_sales.to_string(index=False))

print("\n" + "="*80)
print("KẾT THÚC PHÂN TÍCH GIỚI HẠN DỮ LIỆU")
print("="*80)


TỔNG QUAN VỀ GIỚI HẠN DỮ LIỆU TPC-DS DATASET

1. PHẠM VI THỜI GIAN (Date Ranges)
------------------------------------------------------------
earliest_date latest_date  total_days  start_year  end_year  year_span
   1900-01-02  2100-01-01       73049        1900      2100        201


2. PHẠM VI GIÁ SẢN PHẨM (Price Ranges)
------------------------------------------------------------
 min_price  max_price  avg_price  median_price  min_wholesale_cost  max_wholesale_cost  avg_wholesale_cost
      0.09      99.99   9.523071          4.22                0.02               87.36            5.750199


3. PHẠM VI GIAO DỊCH BÁN HÀNG (Sales Transaction Ranges)
------------------------------------------------------------
 min_quantity  max_quantity  avg_quantity  min_sales_price  max_sales_price  avg_sales_price  min_transaction_value  max_transaction_value  avg_transaction_value
            1           100     50.521831              0.0           199.56        37.867092                    0.0   

## 4. Sample Queries

Let's run some analytical queries on the data.

### Query 1: Top 10 Products by Price

In [5]:
query = """
SELECT 
    i_item_id,
    i_product_name,
    i_category,
    i_current_price
FROM item
WHERE i_current_price > 50
ORDER BY i_current_price DESC
LIMIT 10;
"""

result = con.sql(query).df()
print("Top 10 Most Expensive Products (> $50):\n")
result

Top 10 Most Expensive Products (> $50):



,i_item_id,i_product_name,i_category,i_current_price
0,AAAAAAAACDPAAAAA,oughtn steingpri,Books,99.99
1,AAAAAAAAJDHDAAAA,ationprioughteseought,Electronics,99.96
2,AAAAAAAAEBFAAAAA,barbarpriought,Books,99.89
3,AAAAAAAAGHNAAAAA,ationeseesepri,Home,99.89
4,AAAAAAAAFEMBAAAA,ationpriableation,Sports,99.85
5,AAAAAAAAONIDAAAA,n stantiantieseought,Shoes,99.82
6,AAAAAAAAGNDBAAAA,n stationbaranti,Electronics,99.71
7,AAAAAAAAKIMCAAAA,pribareseoughtought,Sports,99.63
8,AAAAAAAAANAEAAAA,ablen stanticallyought,Children,99.43
9,AAAAAAAABALBAAAA,prioughtn stcally,Shoes,99.42


## 8. Realistic Business Queries

Below are realistic business queries that can be applied to this e-commerce dataset.


### Query 1: Monthly Sales Trend Analysis
Analyze sales performance month-over-month to identify seasonal patterns and growth trends.


In [6]:
# Monthly Sales Trend Analysis
# Purpose: Track sales performance over time to identify seasonal patterns
# How it works: Joins store_sales with date_dim, groups by year and month, calculates revenue metrics

query_monthly_trend = """
SELECT 
    d_year,
    d_moy as month,
    COUNT(DISTINCT ss_sold_date_sk) as active_days,
    COUNT(*) as total_transactions,
    SUM(ss_quantity) as total_units_sold,
    SUM(ss_net_paid) as total_revenue,
    AVG(ss_net_paid) as avg_transaction_value,
    SUM(ss_net_profit) as total_profit,
    (SUM(ss_net_profit) / SUM(ss_net_paid)) * 100 as profit_margin_pct
FROM store_sales
JOIN date_dim ON ss_sold_date_sk = d_date_sk
WHERE d_year >= 2000
GROUP BY d_year, d_moy
ORDER BY d_year DESC, d_moy DESC;
"""

result = con.sql(query_monthly_trend).df()
print("Monthly Sales Trend Analysis:\n")
result


Monthly Sales Trend Analysis:



,d_year,month,active_days,total_transactions,total_units_sold,total_revenue,avg_transaction_value,total_profit,profit_margin_pct
0,2003,1,2,6040,295007.0,1.019323e+07,1725.619509,-4864579.84,-47.723614
1,2002,12,31,94130,4652407.0,1.580923e+08,1719.590846,-75973297.56,-48.056291
2,2002,11,30,88443,4354379.0,1.485214e+08,1719.136538,-71524781.17,-48.157908
3,2002,10,31,62093,3064485.0,1.044066e+08,1722.426082,-50146994.74,-48.030493
4,2002,9,30,59539,2944101.0,1.002997e+08,1726.150607,-48705401.51,-48.559864
5,2002,8,31,57907,2854096.0,9.704707e+07,1715.825081,-46659146.66,-48.078884
6,2002,7,31,26128,1293190.0,4.371248e+07,1713.005731,-21877643.35,-50.048964
7,2002,6,30,25783,1272504.0,4.303635e+07,1708.740907,-20772676.81,-48.267749
8,2002,5,31,26743,1317250.0,4.558631e+07,1745.665542,-21352292.07,-46.839264
9,2002,4,30,25314,1248843.0,4.223137e+07,1712.684273,-20545539.72,-48.649950


### Query 2: Customer Lifetime Value (CLV) Analysis
Calculate total value and purchase frequency for each customer to identify high-value segments.


In [7]:
# Customer Lifetime Value Analysis
# Purpose: Identify high-value customers for targeted marketing
# How it works: Aggregates all sales per customer, calculates total spend, frequency, and average order value

query_clv = """
SELECT 
    c_customer_id,
    c_first_name || ' ' || c_last_name as customer_name,
    c_email_address,
    COUNT(DISTINCT ss_sold_date_sk) as purchase_days,
    COUNT(*) as total_orders,
    SUM(ss_quantity) as total_items_purchased,
    SUM(ss_net_paid) as lifetime_value,
    AVG(ss_net_paid) as avg_order_value,
    MIN(d_date) as first_purchase_date,
    MAX(d_date) as last_purchase_date,
    DATEDIFF('day', MIN(d_date), MAX(d_date)) as customer_lifespan_days
FROM customer
JOIN store_sales ON c_customer_sk = ss_customer_sk
JOIN date_dim ON ss_sold_date_sk = d_date_sk
GROUP BY c_customer_id, c_first_name, c_last_name, c_email_address
HAVING SUM(ss_net_paid) > 50000
ORDER BY lifetime_value DESC
LIMIT 20;
"""

result = con.sql(query_clv).df()
print("Top 20 Customers by Lifetime Value:\n")
result


Top 20 Customers by Lifetime Value:



,c_customer_id,customer_name,c_email_address,purchase_days,total_orders,total_items_purchased,lifetime_value,avg_order_value,first_purchase_date,last_purchase_date,customer_lifespan_days
0,AAAAAAAANBFBBAAA,Lanelle Pollard,Lanelle.Pollard@0gCI3ivHhLT.edu,11,122,6460.0,261794.12,2145.853443,1998-03-11,2002-12-25,1750
1,AAAAAAAAIBAHAAAA,Richard Galloway,Richard.Galloway@8h.org,10,114,5786.0,247359.16,2208.563929,1998-02-13,2002-10-11,1701
2,AAAAAAAAHELOAAAA,Gary Watkins,Gary.Watkins@LeC0.com,10,125,6308.0,232246.67,1888.184309,1999-12-25,2002-09-29,1009
3,AAAAAAAAABIGAAAA,Jose Chavarria,Jose.Chavarria@SYgD.edu,10,112,5888.0,226977.63,2063.433000,1998-01-06,2002-01-12,1467
4,AAAAAAAAKCINAAAA,Robert Robbins,Robert.Robbins@zyrMrkhUmHv.org,9,101,5435.0,223051.85,2253.048990,1998-01-13,2001-11-20,1407
5,AAAAAAAACDFKAAAA,Jo Putnam,Jo.Putnam@K5D9KRF7LDxhrTpmC9s.org,10,122,6637.0,221844.19,1833.423058,1998-05-04,2002-12-17,1688
6,AAAAAAAAADPHBAAA,Elizebeth Doherty,Elizebeth.Doherty@CrYe4Li93bA0.com,9,112,6065.0,219415.50,1976.716216,1999-01-25,2002-08-24,1307
7,AAAAAAAALFJNAAAA,Gladys Hicks,Gladys.Hicks@TULTn0.com,9,101,5265.0,219219.17,2192.191700,1998-01-11,2001-08-20,1317
8,AAAAAAAACACIBAAA,Brenda Ortiz,Brenda.Ortiz@prEcVaI2ax4FTC.org,9,107,5608.0,218289.51,2059.335000,1998-09-20,2002-11-27,1529
9,AAAAAAAAGADKAAAA,Bobbi Chisholm,Bobbi.Chisholm@klrlc2T3xghvKNIl7r4.org,8,94,4893.0,215082.60,2288.112766,1998-12-16,2002-12-25,1470


### Query 3: Product Performance by Category and Brand
Analyze which product categories and brands generate the most revenue and profit.


In [8]:
# Product Performance by Category and Brand
# Purpose: Identify best-performing product categories and brands for inventory planning
# How it works: Joins item with store_sales, groups by category and brand, calculates revenue and profit metrics

query_product_performance = """
SELECT 
    i_category,
    i_brand,
    COUNT(DISTINCT i_item_sk) as num_products,
    SUM(ss_quantity) as total_units_sold,
    SUM(ss_net_paid) as total_revenue,
    SUM(ss_net_profit) as total_profit,
    AVG(ss_sales_price) as avg_selling_price,
    (SUM(ss_net_profit) / SUM(ss_net_paid)) * 100 as profit_margin_pct,
    SUM(ss_net_paid) / COUNT(DISTINCT ss_sold_date_sk) as revenue_per_day
FROM item
JOIN store_sales ON i_item_sk = ss_item_sk
GROUP BY i_category, i_brand
HAVING SUM(ss_net_paid) > 1000000
ORDER BY total_revenue DESC
LIMIT 30;
"""

result = con.sql(query_product_performance).df()
print("Top Product Categories and Brands by Revenue:\n")
result


Top Product Categories and Brands by Revenue:



,i_category,i_brand,num_products,total_units_sold,total_revenue,total_profit,avg_selling_price,profit_margin_pct,revenue_per_day
0,Shoes,importoedu pack #2,243,2348242.0,79834494.85,-38014039.80,37.813250,-47.616059,43792.920927
1,Music,edu packscholar #2,243,2295358.0,78220273.54,-37579268.22,37.834418,-48.042875,42907.445716
2,Music,importoscholar #2,253,2259448.0,76552625.36,-37057863.30,37.824020,-48.408351,41992.663390
3,Music,exportischolar #2,238,2243254.0,76547114.96,-36571909.71,37.811955,-47.776993,42012.686586
4,Men,importoimporto #2,251,2189537.0,74500138.28,-35878233.41,37.813547,-48.158613,40911.662976
5,Women,importoamalg #2,238,2178916.0,74344620.87,-35832262.13,37.966286,-48.197518,40781.470581
6,Women,edu packamalg #2,232,2153852.0,73797587.40,-35187689.74,37.943754,-47.681355,40481.397367
7,Children,edu packexporti #2,223,2120877.0,72754245.65,-35059824.80,37.790661,-48.189387,39909.076056
8,Children,importoexporti #2,220,2123321.0,72615957.43,-34696882.65,37.996355,-47.781347,39833.218557
9,Women,amalgamalg #2,221,2094326.0,71521447.74,-34261974.31,37.788986,-47.904475,39232.829259


### Query 4: Return Rate Analysis
Analyze return rates by product category to identify quality or customer satisfaction issues.


In [9]:
# Return Rate Analysis
# Purpose: Identify products with high return rates that may need quality improvements
# How it works: Compares sales vs returns by category, calculates return rate percentage

query_return_analysis = """
SELECT 
    i_category,
    COUNT(DISTINCT ss_item_sk) as products_sold,
    SUM(ss_quantity) as units_sold,
    SUM(COALESCE(sr_return_quantity, 0)) as units_returned,
    SUM(ss_net_paid) as total_sales_revenue,
    SUM(COALESCE(sr_return_amt, 0)) as total_return_amount,
    CASE 
        WHEN SUM(ss_quantity) > 0 
        THEN (SUM(COALESCE(sr_return_quantity, 0)) * 100.0 / SUM(ss_quantity))
        ELSE 0 
    END as return_rate_pct,
    CASE 
        WHEN SUM(ss_net_paid) > 0 
        THEN (SUM(COALESCE(sr_return_amt, 0)) * 100.0 / SUM(ss_net_paid))
        ELSE 0 
    END as return_revenue_pct
FROM item
JOIN store_sales ON i_item_sk = ss_item_sk
LEFT JOIN store_returns ON ss_item_sk = sr_item_sk 
    AND ss_ticket_number = sr_ticket_number 
    AND ss_store_sk = sr_store_sk
GROUP BY i_category
HAVING SUM(ss_quantity) > 10000
ORDER BY return_rate_pct DESC;
"""

result = con.sql(query_return_analysis).df()
print("Return Rate Analysis by Product Category:\n")
result


Return Rate Analysis by Product Category:



,i_category,products_sold,units_sold,units_returned,total_sales_revenue,total_return_amount,return_rate_pct,return_revenue_pct
0,Shoes,1835,14217132.0,117719.0,4.852623e+08,4409864.90,0.828008,0.908759
1,Electronics,1812,13914945.0,114374.0,4.733032e+08,4236427.03,0.821951,0.895077
2,Sports,1783,13821434.0,113482.0,4.717862e+08,4266003.38,0.821058,0.904224
3,Men,1811,13758179.0,112936.0,4.688480e+08,4205334.84,0.820864,0.896951
4,Jewelry,1740,13495244.0,109867.0,4.582620e+08,4250572.50,0.814116,0.927542
5,Home,1807,13782572.0,110691.0,4.718371e+08,4178379.20,0.803123,0.885555
6,Children,1786,13733212.0,109553.0,4.690829e+08,4111726.48,0.797723,0.876546
7,Women,1790,13787179.0,109025.0,4.693431e+08,4077311.62,0.790771,0.868727
8,Books,1733,13627911.0,106501.0,4.643683e+08,4135542.23,0.781492,0.890574
9,None,43,378753.0,2937.0,1.287056e+07,104528.51,0.775439,0.812152


### Query 5: Sales Channel Comparison (Store vs Web vs Catalog)
Compare performance across different sales channels to optimize channel strategy.


In [10]:
# Sales Channel Comparison
# Purpose: Compare performance across store, web, and catalog channels
# How it works: Unions sales data from all three channels and aggregates by channel

query_channel_comparison = """
SELECT 
    'Store' as channel,
    COUNT(*) as total_transactions,
    SUM(ss_quantity) as total_units,
    SUM(ss_net_paid) as total_revenue,
    AVG(ss_net_paid) as avg_transaction_value,
    SUM(ss_net_profit) as total_profit,
    COUNT(DISTINCT ss_customer_sk) as unique_customers
FROM store_sales
UNION ALL
SELECT 
    'Web' as channel,
    COUNT(*) as total_transactions,
    SUM(ws_quantity) as total_units,
    SUM(ws_net_paid) as total_revenue,
    AVG(ws_net_paid) as avg_transaction_value,
    SUM(ws_net_profit) as total_profit,
    COUNT(DISTINCT ws_bill_customer_sk) as unique_customers
FROM web_sales
UNION ALL
SELECT 
    'Catalog' as channel,
    COUNT(*) as total_transactions,
    SUM(cs_quantity) as total_units,
    SUM(cs_net_paid) as total_revenue,
    AVG(cs_net_paid) as avg_transaction_value,
    SUM(cs_net_profit) as total_profit,
    COUNT(DISTINCT cs_bill_customer_sk) as unique_customers
FROM catalog_sales
ORDER BY total_revenue DESC;
"""

result = con.sql(query_channel_comparison).df()
print("Sales Channel Performance Comparison:\n")
result


Sales Channel Performance Comparison:



,channel,total_transactions,total_units,total_revenue,avg_transaction_value,total_profit,unique_customers
0,Store,2880404,138963631.0,4.736735e+09,1721.955739,-2.281617e+09,90858
1,Catalog,1441548,72414337.0,3.294800e+09,2297.015381,-3.629909e+08,79641
2,Web,719384,36284018.0,1.651211e+09,2295.820411,-1.860938e+08,45161


### Query 6: Top Selling Products by Quarter
Identify best-selling products each quarter for inventory and marketing planning.


In [11]:
# Top Selling Products by Quarter
# Purpose: Track seasonal product performance for inventory and marketing decisions
# How it works: Joins sales with date_dim and item, groups by quarter and product, ranks by revenue

query_top_products_quarterly = """
SELECT 
    d_year,
    d_quarter_name as quarter,
    i_item_id,
    i_product_name,
    i_category,
    i_brand,
    SUM(ss_quantity) as units_sold,
    SUM(ss_net_paid) as revenue,
    AVG(ss_sales_price) as avg_price,
    COUNT(DISTINCT ss_customer_sk) as unique_customers
FROM store_sales
JOIN date_dim ON ss_sold_date_sk = d_date_sk
JOIN item ON ss_item_sk = i_item_sk
WHERE d_year >= 2000
GROUP BY d_year, d_quarter_name, i_item_id, i_product_name, i_category, i_brand
HAVING SUM(ss_net_paid) > 50000
ORDER BY d_year DESC, d_quarter_name, revenue DESC
LIMIT 50;
"""

result = con.sql(query_top_products_quarterly).df()
print("Top Selling Products by Quarter:\n")
result


Top Selling Products by Quarter:



,d_year,quarter,i_item_id,i_product_name,i_category,i_brand,units_sold,revenue,avg_price,unique_customers
0,2002,2002Q1,AAAAAAAAKEFEAAAA,n stpriationationought,Sports,corpmaxi #10,1301.0,83612.86,57.667000,19
1,2002,2002Q1,AAAAAAAAIHEAAAAA,callyeseoughtought,None,None,1422.0,56630.08,41.484545,22
2,2002,2002Q1,AAAAAAAAHFEDAAAA,n stn stpripriought,Children,edu packexporti #2,889.0,54465.33,65.272857,14
3,2002,2002Q1,AAAAAAAAMJABAAAA,eseantiableese,Electronics,scholarunivamalg #5,1163.0,53223.12,39.594783,23
4,2002,2002Q1,AAAAAAAAKBIDAAAA,esecallyprieseought,Men,exportiimporto #1,724.0,53142.17,68.214615,13
5,2002,2002Q1,AAAAAAAAGPEAAAAA,ableationableought,Home,edu packnameless #3,1007.0,52363.35,50.612353,17
6,2002,2002Q1,AAAAAAAAHALBAAAA,n stoughtn stcally,Shoes,amalgedu pack #2,1029.0,51806.53,58.309286,14
7,2002,2002Q1,AAAAAAAAOMACAAAA,barbareseeing,Men,edu packimporto #1,890.0,51779.09,47.511000,20
8,2002,2002Q1,AAAAAAAADDDBAAAA,antioughtn stese,Men,exportiimporto #2,707.0,50725.50,63.407692,13
9,2002,2002Q1,AAAAAAAAKFDEAAAA,eseeseableationought,Books,amalgmaxi #1,890.0,50439.84,51.274000,15


### Query 7: Customer Demographics and Purchase Behavior
Analyze customer demographics to understand purchasing patterns by age, income, and location.


In [12]:
# Customer Demographics and Purchase Behavior
# Purpose: Understand customer segments for targeted marketing campaigns
# How it works: Joins customer demographics with sales data, groups by demographic attributes

query_customer_demographics = """
SELECT 
    cd_gender,
    cd_marital_status,
    cd_education_status,
    ca_state,
    COUNT(DISTINCT c.c_customer_sk) as num_customers,
    SUM(ss_quantity) as total_items_purchased,
    SUM(ss_net_paid) as total_revenue,
    AVG(ss_net_paid) as avg_transaction_value,
    COUNT(*) as total_transactions
FROM customer c
JOIN customer_demographics cd ON c.c_current_cdemo_sk = cd.cd_demo_sk
JOIN customer_address ca ON c.c_current_addr_sk = ca.ca_address_sk
JOIN store_sales ss ON c.c_customer_sk = ss.ss_customer_sk
GROUP BY cd_gender, cd_marital_status, cd_education_status, ca_state
HAVING COUNT(DISTINCT c.c_customer_sk) > 100
ORDER BY total_revenue DESC
LIMIT 30;
"""

result = con.sql(query_customer_demographics).df()
print("Customer Demographics and Purchase Behavior:\n")
result


Customer Demographics and Purchase Behavior:



,cd_gender,cd_marital_status,cd_education_status,ca_state,num_customers,total_items_purchased,total_revenue,avg_transaction_value,total_transactions
0,F,S,Secondary,TX,126,200128.0,6693308.61,1691.938476,4066
1,M,M,4 yr Degree,TX,111,185135.0,6298945.56,1689.178214,3812
2,F,U,College,TX,114,179961.0,6216608.44,1733.577368,3680
3,F,W,Secondary,TX,110,173661.0,6130462.42,1788.871439,3511
4,M,S,Secondary,TX,110,174993.0,6020793.06,1733.101054,3555
5,M,D,Primary,TX,118,174282.0,5923115.45,1713.368658,3540
6,M,U,4 yr Degree,TX,107,169807.0,5908015.23,1756.769322,3428
7,M,M,Secondary,TX,113,170318.0,5827651.61,1759.025539,3385
8,M,S,4 yr Degree,TX,106,160259.0,5579272.45,1766.151456,3229
9,M,D,2 yr Degree,TX,108,165024.0,5538774.62,1709.498340,3323


### Query 8: Promotional Campaign Effectiveness
Analyze which promotions drive the most sales and revenue.


In [13]:
# Promotional Campaign Effectiveness
# Purpose: Measure ROI of promotional campaigns to optimize marketing spend
# How it works: Joins sales with promotion table, compares sales with and without promotions

query_promotion_effectiveness = """
SELECT 
    p_promo_id,
    p_promo_name,
    p_purpose,
    p_discount_active,
    COUNT(*) as transactions_with_promo,
    SUM(ss_quantity) as units_sold,
    SUM(ss_net_paid) as revenue,
    SUM(ss_ext_discount_amt) as total_discount_given,
    AVG(ss_ext_discount_amt) as avg_discount_per_transaction,
    SUM(ss_net_profit) as profit,
    COUNT(DISTINCT ss_customer_sk) as unique_customers
FROM store_sales
JOIN promotion ON ss_promo_sk = p_promo_sk
WHERE p_promo_sk IS NOT NULL
GROUP BY p_promo_id, p_promo_name, p_purpose, p_discount_active
HAVING COUNT(*) > 100
ORDER BY revenue DESC
LIMIT 20;
"""

result = con.sql(query_promotion_effectiveness).df()
print("Promotional Campaign Effectiveness:\n")
result


Promotional Campaign Effectiveness:



,p_promo_id,p_promo_name,p_purpose,p_discount_active,transactions_with_promo,units_sold,revenue,total_discount_given,avg_discount_per_transaction,profit,unique_customers
0,AAAAAAAAGCBAAAAA,ese,Unknown,N,9281,459914.0,16329962.13,1682224.43,186.045613,-7258916.39,8506
1,AAAAAAAAFCBAAAAA,pri,Unknown,N,9406,467410.0,16053534.56,1814394.84,197.517400,-7583880.25,8633
2,AAAAAAAAECAAAAAA,cally,Unknown,N,9256,459559.0,16021380.99,1720384.65,190.350149,-7345789.26,8452
3,AAAAAAAAGIAAAAAA,ese,Unknown,N,9295,461240.0,15883460.61,1860145.69,204.884424,-7361159.66,8526
4,AAAAAAAAGNAAAAAA,None,Unknown,N,9381,462876.0,15860397.14,1792347.50,195.777990,-7670973.65,8585
5,AAAAAAAABOAAAAAA,anti,None,None,9211,454331.0,15859090.92,1781404.36,197.692194,-7664526.79,8443
6,AAAAAAAANABAAAAA,n st,Unknown,N,9193,455105.0,15856844.65,1531005.42,170.833008,-7088615.17,8466
7,AAAAAAAAOOAAAAAA,eing,Unknown,N,9204,452678.0,15842141.20,1628077.15,181.199460,-7094127.92,8452
8,AAAAAAAADBAAAAAA,n st,Unknown,N,9219,454677.0,15840068.87,1769726.65,196.461662,-7348728.49,8477
9,AAAAAAAALMAAAAAA,pri,Unknown,N,9234,454485.0,15823819.51,1760254.21,195.345046,-7192197.58,8482


### Query 9: Store Performance Analysis
Compare performance across different store locations.


In [14]:
# Store Performance Analysis
# Purpose: Identify top and underperforming stores for operational optimization
# How it works: Joins store_sales with store and store address, aggregates sales metrics by store

query_store_performance = """
SELECT 
    s_store_id,
    s_store_name,
    s_state,
    s_city,
    COUNT(*) as total_transactions,
    COUNT(DISTINCT ss_customer_sk) as unique_customers,
    SUM(ss_quantity) as total_units_sold,
    SUM(ss_net_paid) as total_revenue,
    AVG(ss_net_paid) as avg_transaction_value,
    SUM(ss_net_profit) as total_profit,
    (SUM(ss_net_profit) / SUM(ss_net_paid)) * 100 as profit_margin_pct
FROM store_sales
JOIN store ON ss_store_sk = s_store_sk
GROUP BY s_store_id, s_store_name, s_state, s_city
HAVING COUNT(*) > 1000
ORDER BY total_revenue DESC
LIMIT 30;
"""

result = con.sql(query_store_performance).df()
print("Top Performing Stores:\n")
result


Top Performing Stores:



,s_store_id,s_store_name,s_state,s_city,total_transactions,unique_customers,total_units_sold,total_revenue,avg_transaction_value,total_profit,profit_margin_pct
0,AAAAAAAAKAAAAAAA,bar,TN,Midway,459865,32992,22675433.0,7.728572e+08,1721.220262,-3.711894e+08,-48.028200
1,AAAAAAAAIAAAAAAA,eing,TN,Fairview,459745,33103,22673883.0,7.714603e+08,1718.327562,-3.741804e+08,-48.502863
2,AAAAAAAAHAAAAAAA,ation,TN,Midway,458257,32862,22605797.0,7.709518e+08,1723.201693,-3.711289e+08,-48.139059
3,AAAAAAAACAAAAAAA,able,TN,Midway,457864,32966,22579299.0,7.707160e+08,1724.401284,-3.715262e+08,-48.205332
4,AAAAAAAAEAAAAAAA,ese,TN,Midway,458443,32996,22583479.0,7.703805e+08,1720.705606,-3.698746e+08,-48.011932
5,AAAAAAAABAAAAAAA,ought,TN,Midway,456769,32885,22560354.0,7.683828e+08,1722.661382,-3.701438e+08,-48.171802


### Query 10: Year-over-Year Growth Analysis
Calculate year-over-year growth rates for revenue, transactions, and customer base.


In [15]:
# Year-over-Year Growth Analysis
# Purpose: Track business growth trends to measure company performance
# How it works: Aggregates sales by year, uses window functions to calculate YoY growth

query_yoy_growth = """
WITH yearly_metrics AS (
    SELECT 
        d_year,
        COUNT(*) as transactions,
        COUNT(DISTINCT ss_customer_sk) as customers,
        SUM(ss_quantity) as units_sold,
        SUM(ss_net_paid) as revenue,
        SUM(ss_net_profit) as profit
    FROM store_sales
    JOIN date_dim ON ss_sold_date_sk = d_date_sk
    WHERE d_year >= 1998
    GROUP BY d_year
)
SELECT 
    d_year,
    transactions,
    customers,
    units_sold,
    revenue,
    profit,
    LAG(revenue) OVER (ORDER BY d_year) as prev_year_revenue,
    CASE 
        WHEN LAG(revenue) OVER (ORDER BY d_year) > 0 
        THEN ((revenue - LAG(revenue) OVER (ORDER BY d_year)) * 100.0 / LAG(revenue) OVER (ORDER BY d_year))
        ELSE NULL 
    END as revenue_growth_pct,
    CASE 
        WHEN LAG(customers) OVER (ORDER BY d_year) > 0 
        THEN ((customers - LAG(customers) OVER (ORDER BY d_year)) * 100.0 / LAG(customers) OVER (ORDER BY d_year))
        ELSE NULL 
    END as customer_growth_pct
FROM yearly_metrics
ORDER BY d_year DESC;
"""

result = con.sql(query_yoy_growth).df()
print("Year-over-Year Growth Analysis:\n")
result


Year-over-Year Growth Analysis:



,d_year,transactions,customers,units_sold,revenue,profit,prev_year_revenue,revenue_growth_pct,customer_growth_pct
0,2003,6040,532,295007.0,1.019323e+07,-4.864580e+06,9.228462e+08,-98.895457,-98.606418
1,2002,549739,38175,27117113.0,9.228462e+08,-4.457128e+08,9.179189e+08,0.536788,0.664504
2,2001,546314,37923,26931787.0,9.179189e+08,-4.416141e+08,9.329288e+08,-1.608897,-0.860086
3,2000,553970,38252,27354309.0,9.329288e+08,-4.494284e+08,9.126216e+08,2.225154,1.442665
4,1999,544084,37708,26794099.0,9.126216e+08,-4.400867e+08,9.277716e+08,-1.632944,-0.744913
5,1998,550407,37991,27182462.0,9.277716e+08,-4.460023e+08,NaN,NaN,NaN


### Query 2: Sales by Year

In [16]:
query = """
SELECT 
    d_year,
    COUNT(*) as num_sales,
    SUM(ss_net_paid) as total_revenue,
    AVG(ss_net_paid) as avg_sale_amount
FROM store_sales
JOIN date_dim ON ss_sold_date_sk = d_date_sk
GROUP BY d_year
ORDER BY d_year DESC;
"""

result = con.sql(query).df()
print("Sales Summary by Year:\n")
result

Sales Summary by Year:



,d_year,num_sales,total_revenue,avg_sale_amount
0,2003,6040,1.019323e+07,1725.619509
1,2002,549739,9.228462e+08,1719.107685
2,2001,546314,9.179189e+08,1720.889370
3,2000,553970,9.329288e+08,1723.949043
4,1999,544084,9.126216e+08,1718.112003
5,1998,550407,9.277716e+08,1725.935715


### Query 3: Top Customers by Spending

In [17]:
query = """
SELECT 
    c_customer_id,
    c_first_name,
    c_last_name,
    c_email_address,
    SUM(ss_net_paid) as total_spent,
    COUNT(DISTINCT ss_sold_date_sk) as num_purchase_days
FROM customer
JOIN store_sales ON c_customer_sk = ss_customer_sk
GROUP BY c_customer_id, c_first_name, c_last_name, c_email_address
ORDER BY total_spent DESC
LIMIT 10;
"""

result = con.sql(query).df()
print("Top 10 Customers by Total Spending:\n")
result

Top 10 Customers by Total Spending:



,c_customer_id,c_first_name,c_last_name,c_email_address,total_spent,num_purchase_days
0,AAAAAAAANBFBBAAA,Lanelle,Pollard,Lanelle.Pollard@0gCI3ivHhLT.edu,261794.12,11
1,AAAAAAAAIBAHAAAA,Richard,Galloway,Richard.Galloway@8h.org,250675.66,10
2,AAAAAAAAHELOAAAA,Gary,Watkins,Gary.Watkins@LeC0.com,236667.89,10
3,AAAAAAAAABIGAAAA,Jose,Chavarria,Jose.Chavarria@SYgD.edu,227398.83,10
4,AAAAAAAAKCINAAAA,Robert,Robbins,Robert.Robbins@zyrMrkhUmHv.org,226577.62,9
5,AAAAAAAALFJNAAAA,Gladys,Hicks,Gladys.Hicks@TULTn0.com,222907.37,9
6,AAAAAAAAJKONAAAA,Roberta,Krause,Roberta.Krause@mSps1eseRa0NOkof.org,222801.71,11
7,AAAAAAAACDFKAAAA,Jo,Putnam,Jo.Putnam@K5D9KRF7LDxhrTpmC9s.org,221844.19,10
8,AAAAAAAACACIBAAA,Brenda,Ortiz,Brenda.Ortiz@prEcVaI2ax4FTC.org,221437.53,9
9,AAAAAAAAADPHBAAA,Elizebeth,Doherty,Elizebeth.Doherty@CrYe4Li93bA0.com,219415.50,9


### Query 4: Product Categories by Revenue

In [18]:
query = """
SELECT 
    i_category,
    COUNT(DISTINCT i_item_sk) as num_products,
    SUM(ss_quantity) as total_quantity_sold,
    SUM(ss_net_paid) as total_revenue,
    AVG(ss_sales_price) as avg_sale_price
FROM item
JOIN store_sales ON i_item_sk = ss_item_sk
GROUP BY i_category
ORDER BY total_revenue DESC;
"""

result = con.sql(query).df()
print("Product Categories by Revenue:\n")
result

Product Categories by Revenue:



,i_category,num_products,total_quantity_sold,total_revenue,avg_sale_price
0,Music,1860,14447070.0,4.917715e+08,37.872547
1,Shoes,1835,14217132.0,4.852623e+08,37.912646
2,Electronics,1812,13914945.0,4.733032e+08,37.764809
3,Home,1807,13782572.0,4.718371e+08,37.993650
4,Sports,1783,13821434.0,4.717862e+08,37.883625
5,Women,1790,13787179.0,4.693431e+08,37.869650
6,Children,1786,13733212.0,4.690829e+08,37.903361
7,Men,1811,13758179.0,4.688480e+08,37.922391
8,Books,1733,13627911.0,4.643683e+08,37.784154
9,Jewelry,1740,13495244.0,4.582620e+08,37.751868


### Query 5: Customer Demographics

In [19]:
query = """
SELECT 
    ca_state,
    COUNT(DISTINCT c_customer_sk) as num_customers,
    AVG(ss_net_paid) as avg_purchase
FROM customer
JOIN customer_address ON c_current_addr_sk = ca_address_sk
JOIN store_sales ON c_customer_sk = ss_customer_sk
GROUP BY ca_state
ORDER BY num_customers DESC
LIMIT 10;
"""

result = con.sql(query).df()
print("Top 10 States by Customer Count:\n")
result

Top 10 States by Customer Count:



,ca_state,num_customers,avg_purchase
0,TX,7083,1720.026439
1,GA,4496,1719.345631
2,VA,3933,1714.502843
3,KY,3511,1714.354512
4,NC,2988,1719.765544
5,KS,2967,1739.286899
6,IL,2897,1732.417791
7,MO,2894,1709.638048
8,IA,2878,1713.072144
9,None,2792,1740.713734


## 5. Generate Schema Prompt for LLM (Text-to-SQL)

This creates a schema description that can be used with LLMs for text-to-SQL generation.

In [20]:
def generate_schema_prompt(con, table_schema='main', max_tables=None):
    """Generate a schema description for LLM prompts."""
    rows = con.execute(
        '''
        SELECT table_name, column_name, data_type, ordinal_position
        FROM information_schema.columns
        WHERE table_schema = ?
        ORDER BY table_name, ordinal_position
        ''',
        [table_schema],
    ).fetchall()

    tables = {}
    for table_name, column_name, data_type, _ in rows:
        tables.setdefault(str(table_name), []).append((str(column_name), str(data_type)))

    table_names = sorted(tables.keys())
    if max_tables is not None:
        table_names = table_names[:max_tables]

    lines = []
    for t in table_names:
        lines.append(f'TABLE {t} (')
        for col, typ in tables[t]:
            lines.append(f'  {col} {typ}')
        lines.append(')')
        lines.append('')
    return '\n'.join(lines).strip()

# Generate schema for all tables
schema_text = generate_schema_prompt(con)
print("Schema Prompt (first 2000 chars):\n")
print(schema_text[:2000] + "...")

Schema Prompt (first 2000 chars):

TABLE call_center (
  cc_call_center_sk BIGINT
  cc_call_center_id VARCHAR
  cc_rec_start_date DATE
  cc_rec_end_date DATE
  cc_closed_date_sk BIGINT
  cc_open_date_sk BIGINT
  cc_name VARCHAR
  cc_class VARCHAR
  cc_employees BIGINT
  cc_sq_ft BIGINT
  cc_hours VARCHAR
  cc_manager VARCHAR
  cc_mkt_id BIGINT
  cc_mkt_class VARCHAR
  cc_mkt_desc VARCHAR
  cc_market_manager VARCHAR
  cc_division BIGINT
  cc_division_name VARCHAR
  cc_company BIGINT
  cc_company_name VARCHAR
  cc_street_number VARCHAR
  cc_street_name VARCHAR
  cc_street_type VARCHAR
  cc_suite_number VARCHAR
  cc_city VARCHAR
  cc_county VARCHAR
  cc_state VARCHAR
  cc_zip VARCHAR
  cc_country VARCHAR
  cc_gmt_offset DECIMAL(5,2)
  cc_tax_percentage DECIMAL(5,2)
)

TABLE catalog_page (
  cp_catalog_page_sk BIGINT
  cp_catalog_page_id VARCHAR
  cp_start_date_sk BIGINT
  cp_end_date_sk BIGINT
  cp_department VARCHAR
  cp_catalog_number BIGINT
  cp_catalog_page_number BIGINT
  cp_descript

## 6. Custom Query Playground

Use this cell to run your own queries.

In [21]:
# Write your custom query here
custom_query = """
SELECT *
FROM item
LIMIT 5;
"""

result = con.sql(custom_query).df()
result

,i_item_sk,i_item_id,i_rec_start_date,i_rec_end_date,i_item_desc,i_current_price,i_wholesale_cost,i_brand_id,i_brand,i_class_id,...,i_category,i_manufact_id,i_manufact,i_size,i_formulation,i_color,i_units,i_container,i_manager_id,i_product_name
0,1,AAAAAAAABAAAAAAA,1997-10-27,NaT,Powers will not get influences. Electoral port...,27.02,23.23,5003002,exportischolar #2,3,...,Music,52,ableanti,N/A,3663peru009490160959,spring,Tsp,Unknown,6,ought
1,2,AAAAAAAACAAAAAAA,1997-10-27,2000-10-26,"Costs could prove. Industrial, new",1.12,0.38,1001001,amalgamalg #1,1,...,Women,294,esen stable,petite,peru3235638235244700,rosy,Bunch,Unknown,98,able
2,3,AAAAAAAACAAAAAAA,2000-10-27,NaT,"Costs could prove. Industrial, new",7.11,0.38,1001001,brandbrand #4,7,...,Home,294,esen stable,N/A,peru3235638235244700,sienna,Cup,Unknown,18,pri
3,4,AAAAAAAAEAAAAAAA,1997-10-27,1999-10-27,Passengers want etc in a walls. Important year...,1.35,0.85,3002001,importoexporti #1,2,...,Children,479,n stationese,extra large,648935peach259354618,red,Tbl,Unknown,26,ese
4,5,AAAAAAAAEAAAAAAA,1999-10-28,2001-10-26,Passengers want etc in a walls. Important year...,4.00,1.76,2002002,importoimporto #2,2,...,Men,220,barableable,petite,979763908640salmon70,pink,Cup,Unknown,27,anti


In [23]:
import pandas as pd
import random

# 1. Khai báo dữ liệu thực tế trích xuất từ notebook TPC-DS của bạn
categories = ['Music', 'Shoes', 'Electronics', 'Home', 'Sports', 'Books', 'Jewelry', 'Men', 'Women', 'Children']
states = ['TX', 'GA', 'VA', 'KY', 'NC', 'KS', 'IL', 'MO', 'IA']
years = ['1999', '2000', '2001', '2002']
channels = ['Store', 'Web', 'Catalog']
educations = ['Secondary', '4 yr Degree', 'College', 'Primary', '2 yr Degree', 'Advanced Degree']
n_values = ['3', '5', '10']
months = ['1', '6', '12']

# 2. Tập hợp các template từ đơn giản đến phức tạp (đa dạng logic SQL)
templates = [
    # Nhóm 1: Cơ bản & Bộ lọc (Filter)
    "Tổng doanh thu của {cat} tại bang {state} năm {year} là bao nhiêu?",
    "Cho tôi xem danh sách sản phẩm {cat} bán chạy nhất ở {state}.",
    "Trong năm {year}, khách hàng ở {state} mua nhiều {cat} không?",
    "Có bao nhiêu đơn hàng {cat} bị trả lại tại {state} vào năm {year}?",
    "Giá bán trung bình của mặt hàng {cat} trong năm {year} là bao nhiêu?",
    
    # Nhóm 2: So sánh & Đối chiếu (Comparison - JOIN/Window Functions)
    "Năm {year}, doanh thu từ kênh {ch1} cao hơn hay thấp hơn kênh {ch2}?",
    "So sánh lợi nhuận của {cat} giữa kênh {ch1} và {ch2} tại {state}.",
    "Doanh số của {cat} tại {state} năm {year} so với năm trước đó tăng bao nhiêu?",
    "Tỷ lệ trả hàng của {cat} ở {state} có cao hơn mức trung bình hệ thống không?",
    
    # Nhóm 3: Xếp hạng & Top-N (Ranking - ORDER BY/LIMIT)
    "Cho tôi biết top {n} khách hàng chi tiêu nhiều nhất cho mặt hàng {cat} tại {state}.",
    "Tìm {n} sản phẩm {cat} có lợi nhuận thấp nhất trong năm {year}.",
    "Cửa hàng nào ở {state} đứng đầu về doanh số {cat} vào tháng {month}?",
    "Liệt kê {n} thương hiệu {cat} được ưa chuộng nhất bởi khách hàng ở {state}.",
    
    # Nhóm 4: Nhân khẩu học & Hành vi (Demographics - JOIN customer tables)
    "Thống kê tổng số lượng {cat} được mua bởi những khách hàng có trình độ {edu}.",
    "Khách hàng có bằng {edu} tại {state} thường mua {cat} qua kênh nào?",
    "Trong quý 4 năm {year}, nhóm khách hàng {edu} chi bao nhiêu tiền cho {cat}?",
    "Tìm email của các khách hàng VIP ở {state} đã mua đồ {cat} trên {money} đô.",
    
    # Nhóm 5: Câu hỏi ngôn ngữ tự nhiên (Natural/Slang - Cho Voice-bot)
    "Check giùm tôi tồn kho của {cat} ở chi nhánh {state}.",
    "Năm ngoái dân ở {state} mua {cat} nhiều hay ít?",
    "Hàng {cat} trả lại ở {state} có nhiều không, xem số liệu năm {year} thử.",
    "Ai mua đồ {cat} nhiều nhất ở {state} vậy?",
    "Năm {year} thì kênh {ch1} mang về bao nhiêu tiền?"
]

all_queries = []

# 3. Cơ chế sinh mẫu ngẫu nhiên để đạt 300 câu duy nhất
random.seed(42) # Đảm bảo kết quả có thể tái lập
while len(all_queries) < 300:
    temp = random.choice(templates)
    # Điền các giá trị vào chỗ trống
    query = temp.format(
        cat=random.choice(categories),
        state=random.choice(states),
        year=random.choice(years),
        ch1=random.choice(channels),
        ch2=random.choice([c for c in channels if c != 'Store']), 
        edu=random.choice(educations),
        n=random.choice(n_values),
        month=random.choice(months),
        money=random.choice(['1000', '5000', '10000'])
    )
    if query not in all_queries:
        all_queries.append(query)

# 4. Xuất file CSV
df_voice = pd.DataFrame(all_queries, columns=['Câu truy vấn giọng nói'])
df_voice.to_csv('voice_samples_300.csv', index=False, encoding='utf-8-sig')

print(f"Hoàn tất! Đã tạo xong {len(df_voice)} câu mẫu đa dạng vào file 'voice_samples_300.csv'.")
print("-" * 30)
print("Ví dụ 5 câu đầu tiên:")
print(df_voice.head())

Hoàn tất! Đã tạo xong 300 câu mẫu đa dạng vào file 'voice_samples_300.csv'.
------------------------------
Ví dụ 5 câu đầu tiên:
                              Câu truy vấn giọng nói
0               Ai mua đồ Shoes nhiều nhất ở TX vậy?
1   Check giùm tôi tồn kho của Shoes ở chi nhánh IL.
2  Tổng doanh thu của Women tại bang KY năm 2002 ...
3  Thống kê tổng số lượng Books được mua bởi nhữn...
4  Cửa hàng nào ở NC đứng đầu về doanh số Books v...


## 7. Close Connection

In [22]:
con.close()
print("✓ Connection closed.")

✓ Connection closed.
